In [ ]:
import pandas as pd

# Lendo o CSV do DATASUS
# Parâmetros comuns do TabNet: sep=';', encoding='latin1'
# Geralmente as primeiras 3 ou 4 linhas são texto inútil, e as últimas também.
# Vamos tentar ler o arquivo bruto primeiro para ver como ele está.
try:
    df_internacoes = pd.read_csv(
        "../data/raw/cnes_leitos_sp.csv", 
        sep=';', 
        encoding='latin1',
        skiprows=3, # Ajuste esse número dependendo de quantas linhas de título tem no seu CSV
        skipfooter=10, # Ajuste para cortar as notas de rodapé do DATASUS
        engine='python' # Necessário quando usamos skipfooter
    )
    
    print("Base carregada com sucesso!")
    display(df_internacoes.head())
    
except Exception as e:
    print(f"Erro ao carregar a base: {e}")

Base carregada com sucesso!


,110003 CABIXI,8,jul/26
0,110004 CACOAL,571,jul/26
1,110005 CEREJEIRAS,40,jul/26
2,110006 COLORADO DO OESTE,50,jul/26
3,110007 CORUMBIARA,10,jul/26
4,110008 COSTA MARQUES,36,jul/26


In [3]:
import pandas as pd
import io

caminho_cnes = "../data/raw/cnes_leitos_sp.csv"

# 1. Abrimos o arquivo como texto puro para ver o conteúdo e pular o lixo
linhas_validas = []

try:
    with open(caminho_cnes, 'r', encoding='latin1') as arquivo:
        linhas = arquivo.readlines()
        
        # Vamos olhar as 15 primeiras linhas brutas para entender a sujeira
        print("--- CONTEÚDO BRUTO DAS PRIMEIRAS LINHAS ---")
        for i, linha in enumerate(linhas[:15]):
            print(f"Linha {i}: {linha.strip()}")
        print("------------------------------------------\n")
            
except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")

--- CONTEÚDO BRUTO DAS PRIMEIRAS LINHAS ---
Linha 0: Município;Quantidade_existente;Período
Linha 1: 110001 ALTA FLORESTA D'OESTE;55;jul/26
Linha 2: 110002 ARIQUEMES;382;jul/26
Linha 3: 110003 CABIXI;8;jul/26
Linha 4: 110004 CACOAL;571;jul/26
Linha 5: 110005 CEREJEIRAS;40;jul/26
Linha 6: 110006 COLORADO DO OESTE;50;jul/26
Linha 7: 110007 CORUMBIARA;10;jul/26
Linha 8: 110008 COSTA MARQUES;36;jul/26
Linha 9: 110009 ESPIGAO D'OESTE;54;jul/26
Linha 10: 110010 GUAJARA-MIRIM;201;jul/26
Linha 11: 110011 JARU;133;jul/26
Linha 12: 110012 JI-PARANA;384;jul/26
Linha 13: 110013 MACHADINHO D'OESTE;48;jul/26
Linha 14: 110014 NOVA BRASILANDIA D'OESTE;44;jul/26
------------------------------------------



In [4]:
import pandas as pd
import io

caminho_cnes = "../data/raw/cnes_leitos_sp.csv"

# Função para limpar e estruturar os dados do TabNet
def carregar_dados_tabnet(caminho):
    linhas_limpas = []
    
    with open(caminho, 'r', encoding='latin1') as f:
        # Lê todas as linhas do arquivo
        linhas = f.readlines()
        
        for linha in linhas:
            # 1. Ignora linhas vazias ou muito curtas
            if len(linha.strip()) < 5:
                continue
                
            # 2. Ignora o cabeçalho descritivo do MS/DATASUS e os rodapés
            if linha.startswith('"') and not linha.startswith('"1') and not linha.startswith('"2') and not linha.startswith('"3') and not linha.startswith('"M'):
                 # Checamos se começa com aspas, mas não com um número (código IBGE) ou "M" (Município - cabeçalho da tabela)
                 if not linha.startswith('"Município"'):
                     continue
            
            # Limpa as aspas duplas da linha e adiciona à lista
            linhas_limpas.append(linha.replace('"', ''))
            
    # Converte a lista de linhas limpas em um único bloco de texto
    texto_csv = "".join(linhas_limpas)
    
    # Lê o bloco de texto estruturado com o Pandas
    # Usamos sep=';' que é o padrão do arquivo e dtype str para evitar que os códigos percam o 0 à esquerda
    df = pd.read_csv(io.StringIO(texto_csv), sep=';', dtype=str)
    
    return df

# Testando a função
try:
    df_cnes_bruto = carregar_dados_tabnet(caminho_cnes)
    
    print("Base CNES (Leitos) tratada preliminarmente!")
    display(df_cnes_bruto.head())
    
except Exception as e:
    print(f"Erro ao processar: {e}")

Base CNES (Leitos) tratada preliminarmente!


,Município,Quantidade_existente,Período
0,110001 ALTA FLORESTA D'OESTE,55,jul/26
1,110002 ARIQUEMES,382,jul/26
2,110003 CABIXI,8,jul/26
3,110004 CACOAL,571,jul/26
4,110005 CEREJEIRAS,40,jul/26


In [5]:
# Injetando as transformações diretamente no DataFrame atual

# 1. Separando o código IBGE (6 primeiros dígitos) do Nome do Município
df_cnes_bruto[['IBGE', 'Nome_Municipio']] = df_cnes_bruto['Município'].str.split(' ', n=1, expand=True)

# 2. Removendo a coluna antiga que estava misturada
df_cnes_bruto.drop(columns=['Município'], inplace=True)

# 3. Convertendo a coluna de leitos para formato numérico (inteiro) direto na coluna
df_cnes_bruto['Quantidade_existente'] = pd.to_numeric(df_cnes_bruto['Quantidade_existente'], errors='coerce').fillna(0).astype(int)

# 4. Filtrando apenas o estado de São Paulo (IBGE começando com 35)
# Se o seu arquivo só tiver Rondônia, essa tabela ficará vazia. Nesse caso, você precisará baixar o CSV de SP no TabNet.
df_cnes_bruto = df_cnes_bruto[df_cnes_bruto['IBGE'].str.startswith('35', na=False)]

# 5. Reordenando as colunas para ficar organizado
df_cnes_bruto = df_cnes_bruto[['IBGE', 'Nome_Municipio', 'Período', 'Quantidade_existente']]

print("Dados limpos, estruturados e filtrados!")
display(df_cnes_bruto.head())

Dados limpos, estruturados e filtrados!


,IBGE,Nome_Municipio,Período,Quantidade_existente
2250,350010,ADAMANTINA,jul/26,194
2251,350030,AGUAI,jul/26,24
2252,350050,AGUAS DE LINDOIA,jul/26,42
2253,350070,AGUDOS,jul/26,36
2254,350100,ALTINOPOLIS,jul/26,21


In [7]:
import glob

# Pega apenas o primeiro arquivo da lista para teste
caminho_sih_arquivos = "../data/raw/sih_2023_01.csv" 
arquivos_sih = glob.glob(caminho_sih_arquivos)

if len(arquivos_sih) > 0:
    df_teste = carregar_dados_tabnet(arquivos_sih[0])

    print("--- COLUNAS ENCONTRADAS NO ARQUIVO SIH ---")
    print(df_teste.columns.tolist())
    
    print("\n--- PRIMEIRAS LINHAS ---")
    display(df_teste.head())
else:
    print("Nenhum arquivo encontrado. Verifique se o caminho ou o nome (internacoes_*.csv) está correto.")

--- COLUNAS ENCONTRADAS NO ARQUIVO SIH ---
['Município', 'AIH_aprovadas', 'Internações', 'Dias_permanência', 'Média_permanência', 'Óbitos', 'Taxa_mortalidade', 'Período']

--- PRIMEIRAS LINHAS ---


,Município,AIH_aprovadas,Internações,Dias_permanência,Média_permanência,Óbitos,Taxa_mortalidade,Período
0,350010 ADAMANTINA,479,397,4627,"11,7",29,"7,3",01/01/2023
1,350030 AGUAI,7,7,11,"1,6",-,-,01/01/2023
2,350050 AGUAS DE LINDOIA,107,107,364,"3,4",9,"8,41",01/01/2023
3,350070 AGUDOS,73,73,135,"1,8",1,"1,37",01/01/2023
4,350100 ALTINOPOLIS,93,93,184,2,2,"2,15",01/01/2023


In [8]:
import pandas as pd
import glob
import numpy as np
import io

# Recriando a função carregar_dados_tabnet aqui para garantir que está tudo na mesma célula
def carregar_dados_tabnet(caminho):
    linhas_limpas = []
    with open(caminho, 'r', encoding='latin1') as f:
        linhas = f.readlines()
        for linha in linhas:
            if len(linha.strip()) < 5: continue
            if linha.startswith('"') and not linha.startswith('"1') and not linha.startswith('"2') and not linha.startswith('"3') and not linha.startswith('"M'):
                 if not 'Município' in linha:
                     continue
            linhas_limpas.append(linha.replace('"', ''))
    
    texto_csv = "".join(linhas_limpas)
    df = pd.read_csv(io.StringIO(texto_csv), sep=';', dtype=str)
    return df

# 1. Processamento em Lote dos 36 arquivos SIH (Internações)
caminho_sih_arquivos = "../data/raw/internacoes_*.csv" # Mude se o seu padrão for diferente (ex: sih_internacoes_*.csv)
arquivos_sih = glob.glob(caminho_sih_arquivos)

lista_dfs = []

print(f"Iniciando processamento de {len(arquivos_sih)} arquivos...")

for arquivo in arquivos_sih:
    df_temp = carregar_dados_tabnet(arquivo)
    
    # Truque 1: Renomear a primeira coluna na marra para 'Municipio_Bruto', ignorando espaços ocultos
    colunas_atuais = df_temp.columns.tolist()
    if len(colunas_atuais) > 0:
        df_temp = df_temp.rename(columns={colunas_atuais[0]: 'Municipio_Bruto'})
    
    # Truque 2: Separar o IBGE do Nome, pegando a coluna renomeada
    if 'Municipio_Bruto' in df_temp.columns:
        df_temp[['IBGE', 'Nome_Municipio']] = df_temp['Municipio_Bruto'].str.split(' ', n=1, expand=True)
        df_temp.drop(columns=['Municipio_Bruto'], inplace=True)
    
        # Filtrando apenas SP por precaução (IBGE começa com 35)
        # O na=False evita erro caso a linha esteja vazia
        df_temp = df_temp[df_temp['IBGE'].str.startswith('35', na=False)]
        
        # Truque 3: Limpeza dos traços '-' (DATASUS usa '-' para representar zero)
        # Identificando a coluna de internações (geralmente é a que tem "Interna" no nome)
        coluna_internacao = [col for col in df_temp.columns if 'Interna' in col]
        
        if len(coluna_internacao) > 0:
            nome_col = coluna_internacao[0]
            # Substitui traços por 0 e remove espaços
            df_temp[nome_col] = df_temp[nome_col].astype(str).str.replace('-', '0').str.strip()
            # Converte para número
            df_temp['Total_Internacoes'] = pd.to_numeric(df_temp[nome_col], errors='coerce').fillna(0).astype(int)
            # Remove a coluna original suja
            df_temp.drop(columns=[nome_col], inplace=True)
        
        # Opcional: Pegando a Média de Permanência (Importante para o modelo)
        col_permanencia = [col for col in df_temp.columns if 'Média_permanência' in col]
        if len(col_permanencia) > 0:
             nome_col_perm = col_permanencia[0]
             # O Tabnet usa vírgula para decimal. Trocamos para ponto e convertemos para float.
             df_temp[nome_col_perm] = df_temp[nome_col_perm].astype(str).str.replace('-', '0').str.replace(',', '.')
             df_temp['Media_Permanencia_Dias'] = pd.to_numeric(df_temp[nome_col_perm], errors='coerce').fillna(0.0)
             df_temp.drop(columns=[nome_col_perm], inplace=True)
        
        lista_dfs.append(df_temp)

# Empilhando todos os meses em uma única base de dados massiva
if len(lista_dfs) > 0:
    df_sih_historico = pd.concat(lista_dfs, ignore_index=True)
    print("\n--- PROCESSAMENTO CONCLUÍDO! ---")
    print(f"Total de registros na base histórica: {len(df_sih_historico)}")
    display(df_sih_historico.head())
else:
    print("Erro: Nenhum arquivo foi processado com sucesso.")

Iniciando processamento de 0 arquivos...
Erro: Nenhum arquivo foi processado com sucesso.


In [9]:
import pandas as pd
import glob
import numpy as np
import io
import os

# Função para limpar e estruturar os dados do TabNet (Mesma de antes)
def carregar_dados_tabnet(caminho):
    linhas_limpas = []
    try:
        with open(caminho, 'r', encoding='latin1') as f:
            linhas = f.readlines()
            for linha in linhas:
                if len(linha.strip()) < 5: continue
                # Ignora cabeçalhos do TabNet, preservando a linha "Município..." e os dados "35..."
                if linha.startswith('"') and not linha.startswith('"1') and not linha.startswith('"2') and not linha.startswith('"3') and not linha.startswith('"M'):
                     if not 'Município' in linha:
                         continue
                linhas_limpas.append(linha.replace('"', ''))
        
        texto_csv = "".join(linhas_limpas)
        df = pd.read_csv(io.StringIO(texto_csv), sep=';', dtype=str)
        return df
    except Exception as e:
        print(f"Erro ao ler o arquivo {caminho}: {e}")
        return pd.DataFrame() # Retorna DataFrame vazio em caso de erro

# 1. Caminho ajustado para o padrão do Windows
# O 'r' antes das aspas é muito importante no Windows!
caminho_base = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw"
caminho_sih_arquivos = os.path.join(caminho_base, "sih_*.csv") 

arquivos_sih = glob.glob(caminho_sih_arquivos)
lista_dfs = []

print(f"Iniciando processamento de {len(arquivos_sih)} arquivos de internações...")

for arquivo in arquivos_sih:
    df_temp = carregar_dados_tabnet(arquivo)
    
    if df_temp.empty:
        continue

    # Truque 1: Renomear a primeira coluna (Município) ignorando espaços ocultos
    colunas_atuais = df_temp.columns.tolist()
    if len(colunas_atuais) > 0:
        df_temp = df_temp.rename(columns={colunas_atuais[0]: 'Municipio_Bruto'})
    
    # Truque 2: Separar IBGE e Nome
    if 'Municipio_Bruto' in df_temp.columns:
        df_temp[['IBGE', 'Nome_Municipio']] = df_temp['Municipio_Bruto'].str.split(' ', n=1, expand=True)
        df_temp.drop(columns=['Municipio_Bruto'], inplace=True)
    
        # Filtro de Segurança (SP)
        df_temp = df_temp[df_temp['IBGE'].str.startswith('35', na=False)]
        
        # Truque 3: Limpeza da coluna de Internações
        coluna_internacao = [col for col in df_temp.columns if 'Interna' in col]
        if len(coluna_internacao) > 0:
            nome_col = coluna_internacao[0]
            df_temp[nome_col] = df_temp[nome_col].astype(str).str.replace('-', '0').str.strip()
            df_temp['Total_Internacoes'] = pd.to_numeric(df_temp[nome_col], errors='coerce').fillna(0).astype(int)
            df_temp.drop(columns=[nome_col], inplace=True)
        
        # Opcional: Pegando a Média de Permanência
        col_permanencia = [col for col in df_temp.columns if 'Média_permanência' in col]
        if len(col_permanencia) > 0:
             nome_col_perm = col_permanencia[0]
             df_temp[nome_col_perm] = df_temp[nome_col_perm].astype(str).str.replace('-', '0').str.replace(',', '.')
             df_temp['Media_Permanencia_Dias'] = pd.to_numeric(df_temp[nome_col_perm], errors='coerce').fillna(0.0)
             df_temp.drop(columns=[nome_col_perm], inplace=True)
        
        # Adicionando uma coluna para sabermos de qual arquivo o dado veio (útil para ordenar a série temporal)
        nome_arquivo = os.path.basename(arquivo)
        df_temp['Arquivo_Origem'] = nome_arquivo
        
        lista_dfs.append(df_temp)

# 2. Empilhando tudo
if len(lista_dfs) > 0:
    df_sih_historico = pd.concat(lista_dfs, ignore_index=True)
    
    # Vamos manter as colunas mais importantes
    colunas_finais = ['IBGE', 'Nome_Municipio', 'Período', 'Total_Internacoes', 'Media_Permanencia_Dias', 'Arquivo_Origem']
    # Mantém apenas as colunas que realmente existem no DataFrame final
    colunas_existentes = [col for col in colunas_finais if col in df_sih_historico.columns]
    df_sih_historico = df_sih_historico[colunas_existentes]
    
    print("\n--- PROCESSAMENTO CONCLUÍDO! ---")
    print(f"Total de registros na base histórica unificada: {len(df_sih_historico)}")
    display(df_sih_historico.head())
else:
    print("Erro: Nenhum arquivo foi processado. Verifique o caminho e os nomes dos arquivos.")

Iniciando processamento de 36 arquivos de internações...

--- PROCESSAMENTO CONCLUÍDO! ---
Total de registros na base histórica unificada: 11656


,IBGE,Nome_Municipio,Período,Total_Internacoes,Media_Permanencia_Dias,Arquivo_Origem
0,350010,ADAMANTINA,01/01/2023,397.0,11.7,sih_2023_01.csv
1,350030,AGUAI,01/01/2023,7.0,1.6,sih_2023_01.csv
2,350050,AGUAS DE LINDOIA,01/01/2023,107.0,3.4,sih_2023_01.csv
3,350070,AGUDOS,01/01/2023,73.0,1.8,sih_2023_01.csv
4,350100,ALTINOPOLIS,01/01/2023,93.0,2.0,sih_2023_01.csv


In [10]:
import pandas as pd
import html
import io

# Caminho exato do seu arquivo
caminho_ibge_bruto = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\ibge_populacao_sp.csv" 

def limpar_csv_ibge(caminho):
    linhas_limpas = []
    
    with open(caminho, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
        for linha in linhas:
            if ',' not in linha:
                continue
            # Remove entidades HTML estranhas
            linha_decodificada = html.unescape(linha)
            linhas_limpas.append(linha_decodificada)
            
    texto_csv = "".join(linhas_limpas)
    df = pd.read_csv(io.StringIO(texto_csv), sep=',', dtype=str)
    return df

# 1. Carregando e limpando
df_ibge = limpar_csv_ibge(caminho_ibge_bruto)

# 2. Renomeando e ajustando colunas
colunas_atuais = df_ibge.columns.tolist()

col_codigo = [col for col in colunas_atuais if 'Código' in col or 'C&oacute;digo' in col]
col_populacao = [col for col in colunas_atuais if 'População estimada' in col or 'Popula&ccedil;&atilde;o estimada' in col]

if col_codigo and col_populacao:
    df_ibge = df_ibge.rename(columns={
        col_codigo[0]: 'IBGE_Completo', 
        col_populacao[0]: 'Populacao'
    })
    
    df_ibge = df_ibge[['IBGE_Completo', 'Populacao']]
    
    # DATASUS usa 6 dígitos; IBGE usa 7. Cortamos o último dígito para igualar.
    df_ibge['IBGE'] = df_ibge['IBGE_Completo'].str[:6]
    df_ibge['Populacao'] = pd.to_numeric(df_ibge['Populacao'], errors='coerce').fillna(0).astype(int)
    
    # Mantém apenas as colunas úteis
    df_ibge = df_ibge[['IBGE', 'Populacao']]
    
    print("--- BASE DO IBGE LIMPA COM SUCESSO! ---")
    display(df_ibge.head())
else:
    print("Erro: Colunas não encontradas. Verifique o cabeçalho.")

--- BASE DO IBGE LIMPA COM SUCESSO! ---


,IBGE,Populacao
0,350010,35673
1,350020,4505
2,350030,32886
3,350040,7463
4,350050,18257


In [11]:
import numpy as np
import os

# 1. Garante que as bases de referência (CNES e IBGE) estão prontas
# df_sih_historico (Os 36 meses, feito anteriormente)
# df_cnes_limpo (Os leitos, feito anteriormente)
# df_ibge (Feito na Célula 1 acima)

# 2. O Grande Merge: Junta SIH com CNES
df_master = pd.merge(df_sih_historico, df_cnes_limpo, on='IBGE', how='left')
df_master['Quantidade_Leitos'] = df_master['Quantidade_Leitos'].fillna(0)

# 3. Adiciona os dados do IBGE
df_master = pd.merge(df_master, df_ibge, on='IBGE', how='left')

# 4. Cálculo dos Indicadores do HealthOps AI
# IPA - Índice de Pressão Assistencial (Internações / Leitos)
df_master['Ocupacao_Bruta'] = (df_master['Total_Internacoes'] / df_master['Quantidade_Leitos'].replace(0, np.nan)) * 100
df_master['Ocupacao_Bruta'] = df_master['Ocupacao_Bruta'].fillna(0)
df_master['IPA'] = df_master['Ocupacao_Bruta'].apply(lambda x: min(x, 100)).round(2)

# Indicador Estratégico: Internações por Mil Habitantes
df_master['Internacoes_por_1K_Hab'] = ((df_master['Total_Internacoes'] / df_master['Populacao'].replace(0, np.nan)) * 1000).round(2)
df_master['Internacoes_por_1K_Hab'] = df_master['Internacoes_por_1K_Hab'].fillna(0)

# 5. Classificação de Risco (Semáforo do Power BI)
df_master['Status_Risco'] = np.select(
    [
        (df_master['IPA'] < 50),
        (df_master['IPA'] >= 50) & (df_master['IPA'] < 80),
        (df_master['IPA'] >= 80)
    ], 
    ['Estável', 'Alerta', 'Crítico']
)

# 6. Salva a base analítica final
caminho_pasta_destino = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\processed"
if not os.path.exists(caminho_pasta_destino):
    os.makedirs(caminho_pasta_destino)

arquivo_saida = os.path.join(caminho_pasta_destino, "healthops_base_analitica_mvp.csv")
# Usar sep=';' facilita a importação no Power BI sem quebrar colunas
df_master.to_csv(arquivo_saida, index=False, sep=';', encoding='utf-8-sig')

print("--- PIPELINE ETAPA 6 CONCLUÍDO ---")
print(f"Arquivo mestre salvo em: {arquivo_saida}")

print("\n--- AMOSTRA DO RANKING CRÍTICO ---")
display(df_master.sort_values(by='IPA', ascending=False).head())

NameError: name 'df_cnes_limpo' is not defined

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import io

# ---------------------------------------------------------
# 1. RECONSTRUINDO AS BASES NA MEMÓRIA (ETAPAS ANTERIORES)
# ---------------------------------------------------------

# --- A. BASE CNES (LEITOS) ---
caminho_cnes = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\cnes_leitos_sp.csv"
linhas_limpas = []
with open(caminho_cnes, 'r', encoding='latin1') as f:
    linhas = f.readlines()
    for linha in linhas:
        if len(linha.strip()) < 5: continue
        if linha.startswith('"') and not linha.startswith('"1') and not linha.startswith('"2') and not linha.startswith('"3') and not linha.startswith('"M'):
            if not 'Município' in linha: continue
        linhas_limpas.append(linha.replace('"', ''))
texto_csv = "".join(linhas_limpas)
df_cnes_bruto = pd.read_csv(io.StringIO(texto_csv), sep=';', dtype=str)

# Limpeza e separação do CNES
colunas_cnes = df_cnes_bruto.columns.tolist()
if len(colunas_cnes) > 0:
    df_cnes_bruto = df_cnes_bruto.rename(columns={colunas_cnes[0]: 'Municipio_Bruto'})
if 'Municipio_Bruto' in df_cnes_bruto.columns:
    df_cnes_bruto[['IBGE', 'Nome_Municipio']] = df_cnes_bruto['Municipio_Bruto'].str.split(' ', n=1, expand=True)
    df_cnes_bruto = df_cnes_bruto[df_cnes_bruto['IBGE'].str.startswith('35', na=False)]
    col_qtde = [col for col in df_cnes_bruto.columns if 'Quantidade' in col]
    if len(col_qtde) > 0:
        df_cnes_bruto['Quantidade_Leitos'] = pd.to_numeric(df_cnes_bruto[col_qtde[0]], errors='coerce').fillna(0).astype(int)
df_cnes_limpo = df_cnes_bruto[['IBGE', 'Quantidade_Leitos']].copy()

# --- B. BASE SIH (INTERNAÇÕES) ---
# Aqui usamos o df_sih_historico que já testamos e funcionou perfeitamente (11.656 linhas).
# (Se a variável ainda estiver na memória, ele vai usá-la. Senão, vai processar os 36 CSVs de novo rapidinho).
if 'df_sih_historico' not in locals():
    print("Base SIH não está na memória. Recarregando os 36 arquivos...")
    # Código rápido para recarregar caso necessário
    caminho_sih = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\sih_*.csv"
    lista_dfs = []
    for arq in glob.glob(caminho_sih):
        ll = []
        with open(arq, 'r', encoding='latin1') as f:
            for l in f.readlines():
                if len(l.strip()) < 5: continue
                if l.startswith('"') and not l.startswith('"1') and not l.startswith('"2') and not l.startswith('"3') and not l.startswith('"M'):
                    if not 'Município' in l: continue
                ll.append(l.replace('"', ''))
        df_t = pd.read_csv(io.StringIO("".join(ll)), sep=';', dtype=str)
        cols = df_t.columns.tolist()
        if len(cols) > 0: df_t = df_t.rename(columns={cols[0]: 'Mb'})
        if 'Mb' in df_t.columns:
            df_t[['IBGE', 'Nome_Municipio']] = df_t['Mb'].str.split(' ', n=1, expand=True)
            df_t = df_t[df_t['IBGE'].str.startswith('35', na=False)]
            c_int = [c for c in df_t.columns if 'Interna' in c]
            if len(c_int) > 0:
                df_t[c_int[0]] = df_t[c_int[0]].astype(str).str.replace('-', '0').str.strip()
                df_t['Total_Internacoes'] = pd.to_numeric(df_t[c_int[0]], errors='coerce').fillna(0).astype(int)
            c_perm = [c for c in df_t.columns if 'Média_permanência' in c]
            if len(c_perm) > 0:
                df_t[c_perm[0]] = df_t[c_perm[0]].astype(str).str.replace('-', '0').str.replace(',', '.')
                df_t['Media_Permanencia_Dias'] = pd.to_numeric(df_t[c_perm[0]], errors='coerce').fillna(0.0)
            df_t['Arquivo_Origem'] = os.path.basename(arq)
            
            # Formatação do Período para data (ex: 'sih_2023_01.csv' -> '2023-01-01')
            ano = arq[-11:-7]
            mes = arq[-6:-4]
            try:
                df_t['Data_Competencia'] = pd.to_datetime(f"{ano}-{mes}-01")
            except:
                df_t['Data_Competencia'] = pd.NaT
                
            lista_dfs.append(df_t[['IBGE', 'Nome_Municipio', 'Data_Competencia', 'Total_Internacoes', 'Media_Permanencia_Dias', 'Arquivo_Origem']])
    df_sih_historico = pd.concat(lista_dfs, ignore_index=True)


# ---------------------------------------------------------
# 2. O GRANDE MERGE (O CORAÇÃO DO MVP)
# ---------------------------------------------------------

# Junta Internações (SIH) com Leitos (CNES)
df_master = pd.merge(df_sih_historico, df_cnes_limpo, on='IBGE', how='left')
df_master['Quantidade_Leitos'] = df_master['Quantidade_Leitos'].fillna(0)

# Junta com a População (IBGE)
# Certifique-se de que a Célula do IBGE que te mandei antes rodou com sucesso!
if 'df_ibge' in locals():
    df_master = pd.merge(df_master, df_ibge, on='IBGE', how='left')
else:
    print("Aviso: A base do IBGE não foi encontrada na memória. Rode a célula de limpeza do IBGE antes.")

# ---------------------------------------------------------
# 3. CÁLCULO DOS INDICADORES HEALTHOPS AI
# ---------------------------------------------------------

# ---------------------------------------------------------
# 3. CÁLCULO DOS INDICADORES HEALTHOPS AI
# ---------------------------------------------------------

# IPA - Índice de Pressão Assistencial
df_master['Ocupacao_Bruta'] = (df_master['Total_Internacoes'] / df_master['Quantidade_Leitos'].replace(0, np.nan)) * 100
df_master['Ocupacao_Bruta'] = df_master['Ocupacao_Bruta'].fillna(0)
# Limita a 100 para o gráfico do dashboard
df_master['IPA'] = df_master['Ocupacao_Bruta'].apply(lambda x: min(x, 100)).round(2)

# Internações por 1K Hab (Se a base do IBGE existir)
if 'Populacao' in df_master.columns:
    df_master['Internacoes_por_1K_Hab'] = ((df_master['Total_Internacoes'] / df_master['Populacao'].replace(0, np.nan)) * 1000).round(2)
    df_master['Internacoes_por_1K_Hab'] = df_master['Internacoes_por_1K_Hab'].fillna(0)

# Status de Risco (CORRIGIDO)
df_master['Status_Risco'] = np.select(
    [
        (df_master['IPA'] < 50),
        (df_master['IPA'] >= 50) & (df_master['IPA'] < 80),
        (df_master['IPA'] >= 80)
    ], 
    ['Estável', 'Alerta', 'Crítico'],
    default='Não Calculado' # <- O segredo para não dar erro de DType está aqui!
)

# ---------------------------------------------------------
# 4. SALVANDO PARA O POWER BI
# ---------------------------------------------------------
caminho_pasta_destino = r"C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\processed"
if not os.path.exists(caminho_pasta_destino):
    os.makedirs(caminho_pasta_destino)

arquivo_saida = os.path.join(caminho_pasta_destino, "healthops_base_analitica_mvp.csv")
df_master.to_csv(arquivo_saida, index=False, sep=';', encoding='utf-8-sig')

print("--- PIPELINE ETAPA 6 CONCLUÍDO COM SUCESSO ---")
print(f"Arquivo mestre salvo em: {arquivo_saida}")
print("\n--- AMOSTRA DO RANKING (OS PIORES CASOS) ---")
display(df_master.sort_values(by=['IPA', 'Total_Internacoes'], ascending=[False, False]).head())

--- PIPELINE ETAPA 6 CONCLUÍDO COM SUCESSO ---
Arquivo mestre salvo em: C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\processed\healthops_base_analitica_mvp.csv

--- AMOSTRA DO RANKING (OS PIORES CASOS) ---


,IBGE,Nome_Municipio,Período,Total_Internacoes,Media_Permanencia_Dias,Arquivo_Origem,Quantidade_Leitos,Populacao,Ocupacao_Bruta,IPA,Internacoes_por_1K_Hab,Status_Risco
9351,355030,SAO PAULO,01/05/2025,65739.0,5.5,sih_2025_05.csv,30990,11904961,212.129719,100.0,5.52,Crítico
10965,355030,SAO PAULO,01/10/2025,65482.0,5.4,sih_2025_10.csv,30990,11904961,211.300419,100.0,5.50,Crítico
9997,355030,SAO PAULO,01/07/2025,65420.0,5.6,sih_2025_07.csv,30990,11904961,211.100355,100.0,5.50,Crítico
10319,355030,SAO PAULO,01/08/2025,64994.0,5.4,sih_2025_08.csv,30990,11904961,209.725718,100.0,5.46,Crítico
7087,355030,SAO PAULO,01/10/2024,64035.0,5.5,sih_2024_10.csv,30990,11904961,206.631171,100.0,5.38,Crítico
